<a href="https://colab.research.google.com/github/yugan243/Qdrant-Essentials/blob/main/Day01_Project_Semantic_Research_Paper_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Semantic Recommendation System for Research Papers

Build a Semantic search engine for research papers. After set this up, we can ask about a specific area and engine will recommend research papers.

### 1. Install Required Dependencies

- `sentence-transformers`: For creating text embeddings
- `transformers`: Provides tokenizer tools aligned with your embedding model
- `qdrant-client`:  Qdrant Python client
- `llama-index-core`: For text processing and chunking utilities
- `llama-index-embeddings-huggingface`: For embedding our chunks for semantic chunking

In [2]:
!pip install -U sentence-transformers transformers qdrant-client llama-index-core llama-index-embeddings-huggingface -q

### 2. Setting Up Our Environment

Before we start building our Research paper recommendation system, we need to import the necessary libraries. These imports will give us access to:

- `SentenceTransformer` and `HuggingFaceEmbedding`: For creating embeddings from text data
- `QdrantClient`: To interact with our vector search engine
- `Document`, `SentenceSplitter`, and `SemanticSplitterNodeParser`: For breaking down text into manageable chunks
- `AutoTokenizer`: Loads the correct tokenizer for your embedding model, letting you chunk by token count and manage model input limits
- `textwrap`: For formatting text output

Let's import these libraries:

In [3]:
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core import Document
from transformers import AutoTokenizer
import textwrap

### 3. Initialize the Text Encoder

Now we are choosing our text encoder, the core component that will convert our reseach paper descriptions and user queries into numerical vectors (embeddings) These embeddings capture semantic meaning of the text and allow us to find similar content.

We're using the `all-MiniLM-L6-v2` model, which produces 384-dimensional embeddings, and can handle up to 256 tokens per input text.

In [4]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The `all-MiniLM-L6-v2` model is a compact transformer-based language model from the MiniLM architecture, optimized for generating sentence embeddings. These embeddings are dense vectors that encapsulate the semantic meaning of sentences or short texts.

### 4. Define Research Paper Documents

Now we will define our research papers dataset. The collection contains 20 research papers, each with detailed descriptions.

In [12]:
documents = [
    {
        "paper_id": "paper_001",
        "title": "Attention Is All You Need",
        "authors": ["Ashish Vaswani", "Noam Shazeer", "Niki Parmar", "Jakob Uszkoreit", "Llion Jones", "Aidan N. Gomez", "Łukasz Kaiser", "Illia Polosukhin"],
        "year": 2017,
        "field": "Natural Language Processing",
        "venue": "NeurIPS",
        "keywords": ["Transformer", "Attention", "Neural Machine Translation", "Parallelization"],
        "section": "Abstract",
        "text": "This groundbreaking research introduces the Transformer, a novel architectural paradigm for sequence modeling that completely departs from the long-standing reliance on Recurrent Neural Networks (RNNs) and Convolutional Neural Networks (CNNs). The central innovation is the 'Attention' mechanism, specifically 'Self-Attention,' which allows the model to weight the importance of different parts of the input sequence regardless of their physical distance. In traditional models like LSTMs or GRUs, information is processed sequentially, which creates a computational bottleneck and makes it difficult for the model to capture long-range dependencies because gradients often vanish over long sequences. The Transformer solves this by allowing for massive parallelization during the training process, as the entire sequence can be processed at once rather than step-by-step. The architecture is built upon an encoder-decoder structure where each layer utilizes Multi-Head Attention. This allows the model to simultaneously attend to information from different representation subspaces at different positions, effectively learning different types of relationships within the data. For instance, one head might focus on syntactic relationships while another focuses on semantic ones. The authors rigorously evaluated the Transformer on two major machine translation benchmarks: WMT 2014 English-to-German and WMT 2014 English-to-French. In both cases, the Transformer achieved new state-of-the-art results, surpassing previous best-performing ensembles while requiring significantly less time and computational power to train. Beyond its efficiency, the Transformer's ability to handle global dependencies has made it the foundational building block for nearly all modern Large Language Models (LLMs). The paper demonstrates that self-attention is not merely a supplementary tool to be used alongside recurrent layers but is, in fact, a powerful and sufficient mechanism on its own. By removing the sequential nature of processing, the Transformer opened the door to scaling AI models to sizes previously thought impossible. This shift has fundamentally redefined the field of Natural Language Processing and has since extended its influence into Computer Vision and Multimodal AI, proving that the principles of attention and parallelization are universal across various domains of machine intelligence."
    },
    {
        "paper_id": "paper_002",
        "title": "ImageNet Classification with Deep Convolutional Neural Networks",
        "authors": ["Alex Krizhevsky", "Ilya Sutskever", "Geoffrey E. Hinton"],
        "year": 2012,
        "field": "Computer Vision",
        "venue": "NeurIPS",
        "keywords": ["CNN", "Deep Learning", "Image Classification", "ReLU", "Dropout"],
        "section": "Abstract",
        "text": "Commonly referred to as the 'AlexNet' paper, this work is widely recognized as the spark that ignited the modern deep learning revolution. The authors present a large-scale deep convolutional neural network (CNN) designed to tackle the ImageNet Large Scale Visual Recognition Challenge (ILSVRC). At the time, computer vision relied heavily on hand-engineered features, but this paper demonstrated that a sufficiently deep and wide neural network could learn superior features directly from raw pixels. The network consists of 60 million parameters and 650,000 neurons, organized into five convolutional layers (some followed by max-pooling) and three fully connected layers, culminating in a 1000-way softmax output. To make the training of such a massive model feasible, the researchers introduced several key technical innovations. First, they utilized Rectified Linear Units (ReLU) as the activation function, which allowed the network to train much faster than those using traditional tanh or sigmoid functions by alleviating the vanishing gradient problem. Second, they implemented a highly optimized GPU-based convolution operation, distributing the workload across two NVIDIA GPUs to handle the immense memory requirements. To combat the severe problem of overfitting inherent in such high-capacity models, the authors employed two primary strategies: data augmentation and 'dropout.' Data augmentation involved generating image translations and horizontal reflections to artificially increase the size of the training set, while dropout involved randomly setting the output of neurons to zero during training to prevent complex co-adaptations. When tested on the ILSVRC-2010 test set, AlexNet achieved a top-5 error rate of 15.3%, compared to the 26.2% achieved by the second-best entry, which used traditional computer vision methods. This staggering gap in performance proved the dominance of deep learning for visual tasks. The paper also provides a detailed analysis of the learned filters, showing that the first layers learn edge and color blobs, while deeper layers capture more complex textures and object parts. This hierarchical feature learning mirrored the human visual system, providing a theoretical grounding for why CNNs work so effectively. The legacy of this paper is immense, as it shifted the focus of the entire AI community toward deep neural networks and hardware-accelerated computing."
    },
    {
        "paper_id": "paper_003",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": ["Jacob Devlin", "Ming-Wei Chang", "Kenton Lee", "Kristina Toutanova"],
        "year": 2018,
        "field": "Natural Language Processing",
        "venue": "NAACL",
        "keywords": ["BERT", "Transformer", "Pre-training", "Bidirectional", "NLP"],
        "section": "Abstract",
        "text": "BERT, which stands for Bidirectional Encoder Representations from Transformers, introduced a fundamental shift in how language models are pre-trained. Before BERT, most state-of-the-art models were unidirectional, meaning they processed text from left-to-right or used shallow concatenation of left-to-right and right-to-left models. However, the authors argue that such approaches are sub-optimal for sentence-level tasks where understanding the full context is crucial. BERT is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both the left and right context in all layers. The pre-training process is based on two novel unsupervised tasks. The first is the 'Masked Language Model' (MLM), where a percentage of the input tokens are randomly masked, and the model's objective is to predict the original vocabulary id of the masked word based only on its context. This allows the representation to fuse the left and right context, which is impossible in standard unidirectional models as it would allow the model to 'see' the target word. The second task is 'Next Sentence Prediction' (NSP), where the model receives two sentences and must determine if the second sentence follows the first in the original text. This task helps the model understand relationships between sentences, which is vital for tasks like Question Answering (QA) and Natural Language Inference (NLI). One of the most significant advantages of BERT is its simplicity in application: the pre-trained model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks without requiring task-specific architectural modifications. In their experiments, the authors showed that BERT obtained new state-of-the-art results on eleven natural language processing tasks, including the GLUE benchmark, SQuAD v1.1, and SQuAD v2.0. The paper emphasizes that the richness of the bidirectional context is the primary driver of these improvements. By providing a publicly available, high-quality pre-trained model, BERT democratized access to advanced NLP, allowing researchers to achieve excellent results on small datasets by leveraging the knowledge embedded in the BERT weights. This work effectively ended the era of training complex, task-specific architectures from scratch and ushered in the current era of transfer learning in NLP."
    },
    {
        "paper_id": "paper_004",
        "title": "Deep Residual Learning for Image Recognition",
        "authors": ["Kaiming He", "Xiangyu Zhang", "Shaoqing Ren", "Jian Sun"],
        "year": 2016,
        "field": "Computer Vision",
        "venue": "CVPR",
        "keywords": ["ResNet", "Residual Learning", "Skip Connections", "Deep CNN"],
        "section": "Abstract",
        "text": "This paper addresses one of the most significant challenges in deep learning: the degradation problem. As researchers tried to build deeper and deeper neural networks to improve performance, they discovered that after a certain point, adding more layers led to higher training error, not because of overfitting, but because the networks became increasingly difficult to optimize. The authors hypothesized that it is easier for a network to learn a residual mapping than to learn an entire underlying mapping from scratch. To test this, they introduced a 'Residual Learning' framework, where layers are designed to learn a residual function $F(x) = H(x) - x$. The output of these layers is then added back to the input $x$ via 'shortcut connections' or 'skip connections.' This means the network only needs to learn the 'difference' or residual between the input and the desired output. If the optimal mapping is closer to an identity mapping, the weights can simply shrink to zero, making it much easier for the solver to find the solution. The beauty of this approach is that skip connections do not add extra parameters or computational complexity. The researchers evaluated their Residual Networks (ResNets) on the ImageNet dataset with depths of up to 152 layers—which was more than eight times deeper than previous state-of-the-art models like VGG. Despite the extreme depth, ResNet-152 was less complex in terms of FLOPs than VGG-16. The results were definitive: ResNets were much easier to optimize and showed significant gains in accuracy from increased depth. The ResNet ensemble won first place in several tracks of the ILSVRC 2015 and COCO 2015 competitions, including ImageNet classification, detection, and localization, as well as COCO detection and segmentation. The paper also demonstrates that the depth of the representations is of central importance for many visual recognition tasks. By solving the optimization issues associated with very deep networks, ResNet became a standard architecture in computer vision, influencing everything from object detection to medical imaging. Its core principle of using skip connections has since been integrated into many other architectures, including the Transformer, highlighting the universal utility of residual learning in modern AI."
    },
    {
        "paper_id": "paper_005",
        "title": "Generative Adversarial Networks",
        "authors": ["Ian Goodfellow", "Jean Pouget-Abadie", "Mehdi Mirza", "Bing Xu", "David Warde-Farley", "Sherjil Ozair", "Aaron Courville", "Yoshua Bengio"],
        "year": 2014,
        "field": "Generative AI",
        "venue": "NeurIPS",
        "keywords": ["GAN", "Generative Models", "Adversarial Training", "Game Theory"],
        "section": "Abstract",
        "text": "In this seminal paper, the authors introduce a radically new framework for training generative models based on game theory, known as Generative Adversarial Networks (GANs). The core idea is to set up a competition between two neural networks: a Generator (G) and a Discriminator (D). The Generator's task is to create synthetic data samples (such as images) from random noise, while the Discriminator's task is to distinguish between real samples from the training dataset and the 'fake' samples produced by the Generator. This setup is mathematically framed as a zero-sum, minimax game where the two models are trained simultaneously. The Discriminator is trained to maximize the probability of assigning the correct label to both real and fake examples, while the Generator is trained to minimize the probability that the Discriminator can detect its fakes. Essentially, the Generator is trying to 'fool' the Discriminator. As training progresses, the Generator becomes increasingly adept at capturing the underlying distribution of the training data, producing samples that are so realistic the Discriminator can no longer tell the difference. Unlike previous generative models, such as Boltzmann machines or Variational Autoencoders, GANs do not require complex Markov chains or explicit density functions, which makes them much more flexible and capable of generating high-fidelity images. The authors provide a theoretical proof that if both models have sufficient capacity, the game has a unique global optimum where the Generator exactly recovers the data generating distribution and the Discriminator is unable to do better than random guessing (a probability of 0.5). Experimental results on datasets like MNIST, the Toronto Face Database (TFD), and CIFAR-10 demonstrated that GANs could produce sharp, realistic images. While GANs are notoriously difficult to train due to issues like mode collapse and training instability, this paper opened an entire sub-field of AI. The adversarial training paradigm has since been extended to numerous applications, including high-resolution image synthesis, style transfer, text-to-image generation, and even drug discovery. By shifting the focus from explicit density estimation to a competitive learning process, the authors provided a powerful new tool for modeling complex, high-dimensional data distributions, cementing GANs as a cornerstone of modern generative artificial intelligence."
    },
    {
        "paper_id": "paper_006",
        "title": "Language Models are Few-Shot Learners",
        "authors": ["Tom B. Brown", "Benjamin Mann", "Nick Ryder", "Melanie Subbiah", "Jared Kaplan", "Prafulla Dhariwal", "Arvind Neelakantan", "Pranav Shyam", "Girish Sastry", "Amanda Askell"],
        "year": 2020,
        "field": "Generative AI",
        "venue": "NeurIPS",
        "keywords": ["GPT-3", "Few-Shot Learning", "Large Language Models", "Zero-Shot"],
        "section": "Abstract",
        "text": "This paper introduces GPT-3, a massive autoregressive language model with 175 billion parameters, which was at the time of publication ten times larger than any previous non-sparse language model. The central thesis of the work is that simply scaling up language models leads to significant improvements in their ability to perform tasks with little to no task-specific data. The authors evaluate GPT-3 in three settings: 'zero-shot' (no examples given), 'one-shot' (one example given), and 'few-shot' (a few examples given, typically 10 to 100), all without any weight updates or fine-tuning. This is a departure from the standard machine learning paradigm of pre-training followed by supervised fine-tuning. GPT-3 was trained on an enormous and diverse corpus of text, including Common Crawl, WebText2, and books. The results across a wide array of NLP tasks—including translation, question-answering, and cloze tasks—showed that GPT-3 achieves strong performance, often rivaling or even surpassing the state-of-the-art models that were specifically fine-tuned for those tasks. For example, on the TriviaQA benchmark, GPT-3 in the few-shot setting outperformed the previous best fine-tuned model. Beyond standard NLP tasks, GPT-3 demonstrated surprising 'emergent' abilities, such as performing basic arithmetic, writing functional code, and generating high-quality news articles that human evaluators struggled to distinguish from those written by people. However, the authors also highlight significant limitations and societal concerns. GPT-3 still struggles with certain types of common-sense reasoning, such as determining if a person would be wet after walking through a sprinkler, and it can exhibit biases present in its training data. The paper discusses the broader impacts of such large-scale models, including their potential for misuse in generating misinformation and their substantial energy consumption during training. Despite these challenges, the work provides compelling evidence that 'scale' is a primary driver of intelligence in language models. By showing that a single, massive model can generalize across hundreds of tasks without being explicitly trained for them, the authors suggest that we are moving toward a future of more general-purpose AI systems. This paper solidified the 'In-Context Learning' paradigm and spurred a massive wave of research into even larger models and more efficient ways to harness their capabilities."
    },
    {
        "paper_id": "paper_007",
        "title": "Playing Atari with Deep Reinforcement Learning",
        "authors": ["Volodymyr Mnih", "Koray Kavukcuoglu", "David Silver", "Alex Graves", "Ioannis Antonoglou", "Daan Wierstra", "Martin Riedmiller"],
        "year": 2013,
        "field": "Reinforcement Learning",
        "venue": "NIPS Deep Learning Workshop",
        "keywords": ["Deep Q-Learning", "DQN", "Atari", "Experience Replay"],
        "section": "Abstract",
        "text": "This landmark paper presents the first successful attempt to combine deep learning with reinforcement learning (RL) to solve complex, high-dimensional tasks directly from raw sensory input. The authors introduce the Deep Q-Network (DQN), an agent that learns to play various Atari 2600 games using only the pixel data as input and the game score as the reward signal. Historically, RL struggled with high-dimensional inputs like images because they required hand-crafted features to make the state space manageable. Furthermore, combining RL with neural networks was known to be unstable due to correlations in the sequence of observations and the fact that small updates to the policy could significantly change the data distribution. To overcome these hurdles, the authors implemented two critical innovations. The first is a 'convolutional' architecture that can automatically learn hierarchical spatial features from the raw game frames. The second, and more vital, is a technique called 'Experience Replay.' The agent's experiences (state, action, reward, next state) are stored in a memory buffer, and training is performed by sampling random mini-batches from this buffer. This breaks the temporal correlations in the data and stabilizes the learning process by allowing the model to learn from a more diverse set of past experiences. The DQN was tested on seven different Atari games, including Pong, Breakout, and Space Invaders. Using the exact same network architecture and hyperparameter settings for all games, the agent achieved superhuman performance on three of them and outperformed all previous RL algorithms on almost all of them. The success of DQN demonstrated that deep learning could be used to automate the feature engineering process in RL, enabling agents to learn directly from their environment. This work effectively birthed the field of Deep Reinforcement Learning, showing that the same principles used in computer vision and speech recognition could be applied to decision-making and control tasks. It paved the way for more advanced systems like AlphaGo and self-driving car algorithms, proving that end-to-end learning from perception to action is a viable path toward building truly autonomous agents."
    },
    {
        "paper_id": "paper_008",
        "title": "Mask R-CNN",
        "authors": ["Kaiming He", "Georgia Gkioxari", "Piotr Dollár", "Ross Girshick"],
        "year": 2017,
        "field": "Computer Vision",
        "venue": "ICCV",
        "keywords": ["Instance Segmentation", "Object Detection", "Mask R-CNN", "RoIAlign"],
        "section": "Abstract",
        "text": "The authors introduce Mask R-CNN, an elegant and powerful extension of the Faster R-CNN framework designed for instance segmentation. Instance segmentation is a challenging task that requires not only detecting the bounding box of every object in an image but also generating a pixel-level mask for each detected instance. While Faster R-CNN was excellent at object detection (predicting bounding boxes and class labels), it was not designed for pixel-perfect localization. Mask R-CNN addresses this by adding a third branch to the network that predicts a binary mask for each Region of Interest (RoI) in parallel with the classification and box regression branches. A major technical contribution of this paper is the introduction of 'RoIAlign.' Previous models used 'RoIPool,' which performed coarse spatial quantization to align features with a fixed-size grid. While sufficient for classification, this quantization introduced misalignments that were detrimental to mask accuracy. RoIAlign avoids any quantization, using bilinear interpolation to compute the exact values of the input features at each location. This seemingly small change led to a massive improvement in mask quality. The architecture is modular and flexible, allowing researchers to swap out different 'backbone' networks (like ResNet or ResNeXt) to balance speed and accuracy. In their experiments on the COCO dataset, Mask R-CNN surpassed all existing single-model entries for instance segmentation, bounding-box object detection, and person keypoint detection. Despite its complexity, Mask R-CNN is relatively easy to train and adds only a small computational overhead to the base Faster R-CNN model, running at approximately 5 frames per second. The paper also demonstrates that training the three tasks (classification, detection, and mask generation) simultaneously leads to better overall performance, as the shared features benefit from the diverse supervision signals. Mask R-CNN has since become a foundational tool in computer vision, used extensively in fields ranging from medical image analysis—where precise tumor segmentation is required—to autonomous driving, where detecting and delineating individual pedestrians and vehicles is crucial for safety. Its success highlight the trend in AI toward unified, multi-task architectures that can handle multiple facets of scene understanding within a single framework."
    },
    {
        "paper_id": "paper_009",
        "title": "Mastering the Game of Go with Deep Neural Networks and Tree Search",
        "authors": ["David Silver", "Aja Huang", "Chris J. Maddison", "Arthur Guez", "Laurent Sifre", "George van den Driessche", "Julian Schrittwieser"],
        "year": 2016,
        "field": "Reinforcement Learning",
        "venue": "Nature",
        "keywords": ["AlphaGo", "Deep RL", "Monte Carlo Tree Search", "Go"],
        "section": "Abstract",
        "text": "The ancient game of Go has long been considered the 'holy grail' of AI research due to its immense complexity, with a search space far larger than that of chess and the difficulty of evaluating board positions using traditional heuristic methods. This paper details the creation of AlphaGo, the first computer program to defeat a professional human player on a full-sized 19x19 board. AlphaGo's architecture is a sophisticated combination of deep convolutional neural networks and Monte Carlo Tree Search (MCTS). The system utilizes two primary networks: the 'policy network,' which is trained to predict the most likely next moves made by expert human players, and the 'value network,' which is trained to predict the winner of the game from a given board configuration. The training process was multi-staged. Initially, the policy network was trained using supervised learning on a dataset of 30 million moves from human expert games. This was followed by a reinforcement learning (RL) stage, where the system played millions of games against slightly older versions of itself to refine its strategy. The value network was then trained on the data generated by these self-play games to evaluate the probability of winning. During actual gameplay, AlphaGo uses MCTS to look ahead and simulate potential game outcomes, using the policy network to narrow the search to promising moves and the value network (combined with fast rollouts) to evaluate the strength of those moves. In a historic 2015 match, AlphaGo defeated the three-time European Champion Fan Hui with a score of 5-0. Subsequent versions of the system went on to defeat Lee Sedol, one of the world's top players, in a globally televised match. This work provided definitive proof that deep reinforcement learning could be applied to problems with extreme strategic depth and a vast branching factor. Beyond the game of Go, the principles developed for AlphaGo—combining high-level intuition (neural networks) with precise calculation (tree search)—have broad implications for complex decision-making tasks in science and engineering, such as protein folding and materials discovery. The paper is a landmark in AI, marking the point where machines surpassed human experts in one of the most intellectually demanding games in history."
    },
    {
        "paper_id": "paper_010",
        "title": "Learning Joint Adversarial-Learned Representations for Multimodal AI",
        "authors": ["Tadas Baltrusaitis", "Amir Zadeh", "Yao-Chung Lim", "Louis-Philippe Morency"],
        "year": 2018,
        "field": "Multimodal AI",
        "venue": "CVPR",
        "keywords": ["Multimodal", "Representation Learning", "Fusion", "Deep Learning"],
        "section": "Abstract",
        "text": "As humans, we interact with the world through a combination of senses—vision, hearing, and touch—and language. Multimodal machine learning aims to replicate this ability by building models that can process, relate, and fuse information from multiple data streams. This paper explores the fundamental challenges of multimodal representation learning, specifically focusing on the 'fusion' problem: how to combine information from different modalities with varying levels of noise and structure. The authors argue that simple concatenation of feature vectors is often insufficient because it fails to capture the complex, non-linear interactions between modalities. To address this, they propose a framework that utilizes adversarial learning to align representations into a shared latent space. The core idea is to train modality-specific encoders that map video, audio, and text into a common distribution. An adversarial discriminator is then used to ensure that the representations from different modalities are indistinguishable in the shared space, effectively 'forcing' the model to find a common language between the signals. This approach helps the model ignore modality-specific noise and focus on the underlying semantic information that is shared across all inputs. The paper also investigates the concept of 'coordinated representations,' where modalities are kept separate but are constrained by their similarity. The researchers evaluate their methods on tasks like multimodal sentiment analysis and emotion recognition, using datasets such as CMU-MOSEI. The results demonstrate that joint adversarial learning leads to more robust performance, especially when one of the modalities is missing or corrupted during testing. The study provides a comprehensive taxonomy of multimodal learning, identifying five core challenges: representation, translation, alignment, fusion, and co-learning. By providing both a theoretical framework and strong empirical evidence, this work has become a key reference for researchers building intelligent systems that can 'understand' the world in a more human-like way. The techniques discussed are now foundational for developing sophisticated conversational AI, autonomous robots, and video analysis tools that must integrate diverse sensory information to make accurate predictions."
    },
    {
        "paper_id": "paper_011",
        "title": "You Only Look Once: Unified, Real-Time Object Detection",
        "authors": ["Joseph Redmon", "Santosh Divvala", "Ross Girshick", "Ali Farhadi"],
        "year": 2016,
        "field": "Computer Vision",
        "venue": "CVPR",
        "keywords": ["YOLO", "Object Detection", "Real-time", "CNN"],
        "section": "Abstract",
        "text": "Before the introduction of YOLO (You Only Look Once), object detection systems were complex, multi-stage pipelines. Models like R-CNN used a 'proposal and classify' approach: they first generated thousands of potential bounding boxes in an image and then ran a heavy classifier on each box to identify objects. While accurate, these systems were incredibly slow and difficult to optimize because each component had to be trained separately. This paper presents a radically different approach: framing object detection as a single regression problem. YOLO uses a single neural network to predict bounding boxes and class probabilities directly from full images in one evaluation. The entire image is divided into an $S \times S$ grid. For each grid cell, the network predicts multiple bounding boxes and confidence scores, as well as class probabilities. Because the detection process is a single network, it can be optimized end-to-end directly on detection performance. The biggest advantage of YOLO is its incredible speed. The standard YOLO model processes images at 45 frames per second, while a smaller version, 'Fast YOLO,' achieves a staggering 155 fps. This made real-time object detection on standard hardware possible for the first time. Another key benefit is that YOLO reasons globally about the image. Because it sees the entire image during training and inference, it captures contextual information about classes and their appearances, which helps it avoid making background errors—a common problem for localized methods like Fast R-CNN. While YOLO's localization accuracy initially lagged behind state-of-the-art systems, especially for small objects, its speed and generalizability were unmatched. The authors showed that YOLO generalizes better than other methods when moving from natural images to artwork, proving that it learns more robust features. This paper fundamentally changed the trajectory of object detection research, shifting the community's focus from pure accuracy to the critical trade-off between speed and performance. YOLO has since gone through many iterations (v2, v3, v4, etc.) and remains the most popular architecture for real-world applications like autonomous driving, drone navigation, and video surveillance where real-time processing is a non-negotiable requirement."
    },
    {
        "paper_id": "paper_012",
        "title": "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale",
        "authors": ["Alexey Dosovitskiy", "Lucas Beyer", "Alexander Kolesnikov", "Dirk Weissenborn", "Xiaohua Zhai", "Thomas Unterthiner", "Mostafa Dehghani", "Matthias Minderer", "Georg Heigold", "Sylvain Gelly", "Jakob Uszkoreit", "Neil Houlsby"],
        "year": 2021,
        "field": "Computer Vision",
        "venue": "ICLR",
        "keywords": ["Vision Transformer", "ViT", "Transformer", "Image Classification"],
        "section": "Abstract",
        "text": "For nearly a decade, Convolutional Neural Networks (CNNs) were the undisputed kings of computer vision, with their success attributed to built-in 'inductive biases' like translation invariance and local connectivity. This paper challenges that dominance by showing that the Transformer architecture, which had already revolutionized NLP, can be applied directly to images with minimal modifications. The authors introduce the Vision Transformer (ViT). Since Transformers are designed for 1D sequences, the researchers adapt images by splitting them into a series of fixed-size patches (e.g., 16x16 pixels). Each patch is linearly projected into an embedding, and a sequence of these 'visual words' is fed into a standard Transformer encoder. To allow the model to perform classification, a special 'class token' is added to the sequence, and position embeddings are added to maintain spatial information. The most surprising finding of the paper is that when trained on mid-sized datasets like ImageNet-1k, ViT performs slightly worse than ResNets of comparable size. However, as the dataset size increases, the Transformer's lack of inductive bias becomes an advantage. When pre-trained on massive datasets like ImageNet-21k (14 million images) or JFT-300M (300 million images), ViT matches or exceeds the performance of state-of-the-art CNNs on multiple benchmarks while being significantly more efficient to train. For example, ViT-Huge attained 88.55% accuracy on ImageNet, setting a new record. The authors argue that while CNNs have a head start because of their architectural assumptions about the structure of images, Transformers are 'purer' learners that can discover the optimal structures directly from data if given enough examples. The paper also provides fascinating visualizations showing that the Transformer's self-attention layers learn to attend to both local and global features in the very first layers, whereas CNNs must build up to global features through many successive layers. This work sparked a paradigm shift in the field, leading to the rapid adoption of Transformers for object detection, segmentation, and multimodal tasks. It suggested that a single architecture—the Transformer—could eventually serve as a universal model for both vision and language, simplifying the AI landscape and enabling more seamless integration of different data types."
    },
    {
        "paper_id": "paper_013",
        "title": "Proximal Policy Optimization Algorithms",
        "authors": ["John Schulman", "Filip Wolski", "Prafulla Dhariwal", "Alec Radford", "Oleg Klimov"],
        "year": 2017,
        "field": "Reinforcement Learning",
        "venue": "arXiv",
        "keywords": ["PPO", "Policy Gradient", "Reinforcement Learning", "Optimization"],
        "section": "Abstract",
        "text": "Deep Reinforcement Learning (RL) has achieved remarkable successes, but many of its most powerful algorithms, such as Vanilla Policy Gradient or TRPO, are difficult to use in practice. They often require extensive hyperparameter tuning, suffer from poor sample efficiency, and can be highly unstable—meaning a single bad update can ruin the agent's performance and prevent it from ever recovering. This paper introduces Proximal Policy Optimization (PPO), a new family of policy gradient methods designed to be more robust, efficient, and easier to implement. The core innovation of PPO is a 'clipped surrogate objective.' In standard policy gradient methods, the objective function is sensitive to the step size; if the step is too large, the policy changes too drastically. PPO solves this by introducing a constraint that prevents the new policy from deviating too far from the old policy. Specifically, it clips the probability ratio between the new and old policies within a small range (typically 0.8 to 1.2). This ensures that updates are conservative and stable, allowing the model to perform multiple epochs of optimization on the same batch of data without the risk of catastrophic collapses. The authors compare PPO against several other RL algorithms, including A2C and TRPO, across a variety of environments. These include continuous control tasks like robotic locomotion (using the MuJoCo simulator) and discrete tasks like playing Atari games. PPO consistently outperformed the other methods in terms of both final performance and sample efficiency. Furthermore, unlike TRPO, which requires complex second-order optimization (Kronecker-factored approximate curvature), PPO only uses first-order optimization, making it much simpler to code and faster to run. Because of its balance between ease of use and state-of-the-art performance, PPO has become the 'go-to' algorithm for RL practitioners and researchers worldwide. It is the default algorithm used by OpenAI for many of their most complex projects, including training the dextrous robotic hand and the Dota 2 agents. This paper is a testament to the idea that sometimes the most impactful research is not just about pushing the absolute limits of performance, but about making powerful tools more accessible and reliable for the broader community."
    },
    {
        "paper_id": "paper_014",
        "title": "DALL-E: Zero-Shot Text-to-Image Generation",
        "authors": ["Aditya Ramesh", "Mikhail Pavlov", "Gabriel Goh", "Scott Gray", "Chelsea Voss", "Alec Radford", "Mark Chen", "Ilya Sutskever"],
        "year": 2021,
        "field": "Generative AI",
        "venue": "ICML",
        "keywords": ["Text-to-Image", "Generative AI", "Transformer", "Zero-Shot"],
        "section": "Abstract",
        "text": "The dream of creating a system that can generate any image from a simple text description has been a long-standing goal of AI. This paper introduces DALL-E, a 12-billion parameter transformer model that brings this dream closer to reality. DALL-E is an extension of the GPT-3 architecture, but instead of being trained solely on text, it is trained on a massive dataset of text-image pairs. The key technical challenge the authors faced was how to represent images in a way that a transformer—which is designed for discrete tokens—could understand. They solved this by using a discrete Variational Autoencoder (dVAE) to compress 256x256 images into a 32x32 grid of image tokens, each belonging to a vocabulary of 8192 possible values. This allows the model to treat an image as a sequence of tokens, just like a sentence. During training, the transformer receives the text tokens followed by the image tokens and learns to predict the next token in the sequence. This unified approach allows DALL-E to perform a wide variety of tasks. It can generate images from scratch based on prompts like 'an armchair in the shape of an avocado,' it can perform 'zero-shot' image-to-image translation, and it can even modify specific parts of an image while maintaining stylistic consistency. One of the most remarkable findings of the paper is the model's ability to generalize to novel concepts. For example, it can correctly render an 'otter in the style of Vermeer' or a 'baby penguin wearing a blue hat,' even if it has never seen such specific combinations in its training data. This suggests that DALL-E has learned a deep understanding of the relationship between linguistic concepts and visual attributes. The authors also highlight the model's 'compositional' nature, as it can combine multiple independent concepts (e.g., color, texture, and shape) to create complex scenes. While DALL-E occasionally fails or produces artifacts, its success marked a turning point in generative AI, proving that the transformer architecture is not limited to text and can be a powerful engine for visual creativity. This work laid the foundation for the explosion of text-to-image models like Stable Diffusion and Midjourney, fundamentally changing how we think about art, design, and content creation."
    },
    {
        "paper_id": "paper_015",
        "title": "Long Short-Term Memory",
        "authors": ["Sepp Hochreiter", "Jürgen Schmidhuber"],
        "year": 1997,
        "field": "Natural Language Processing",
        "venue": "Neural Computation",
        "keywords": ["LSTM", "RNN", "Vanishing Gradient", "Sequence Modeling"],
        "section": "Abstract",
        "text": "In the late 1990s, Recurrent Neural Networks (RNNs) were the primary tool for sequence modeling, but they suffered from a fatal flaw known as the 'vanishing gradient problem.' As gradients were propagated back through time to train the network, they would shrink exponentially, making it impossible for the model to learn relationships between data points that were separated by more than a few steps. This paper introduces the Long Short-Term Memory (LSTM) architecture, a revolutionary solution to this problem that would go on to dominate the field of AI for over two decades. The core idea of the LSTM is the 'constant error carousel,' a memory cell that can maintain its state over indefinitely long periods. The authors introduced a sophisticated set of 'gates'—input, output, and (later) forget gates—that regulate the flow of information into and out of this cell. The input gate decides which new information is worth storing, the forget gate (added in subsequent work) decides which old information is no longer relevant, and the output gate decides which part of the memory should be used to produce the current output. Because the memory cell's state is updated through addition rather than multiplication, gradients can flow through the network for hundreds or even thousands of steps without vanishing. The paper provides a rigorous mathematical analysis of why LSTMs are stable and demonstrates their superiority on several difficult artificial tasks, such as the 'adding problem' and long-distance dependency tasks that were completely unsolvable for standard RNNs. Although the paper was initially met with some skepticism due to its complexity, the LSTM eventually became the backbone of modern technology, powering everything from Google Translate and Siri to early versions of Alexa. It proved that deep learning could handle the temporal nature of speech and language, bridging the gap between simple pattern recognition and complex sequence understanding. While the Transformer has largely superseded the LSTM in recent years for large-scale NLP, the principles introduced in this paper—specifically the idea of using gates to manage internal state—remain fundamental to our understanding of how artificial systems can process and remember information over time. It is one of the most cited papers in the history of computer science, and its legacy continues to influence the design of recurrent systems today."
    },
    {
        "paper_id": "paper_016",
        "title": "Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation",
        "authors": ["Kyunghyun Cho", "Bart van Merriënboer", "Caglar Gulcehre", "Dzmitry Bahdanau", "Fethi Bougares", "Holger Schwenk", "Yoshua Bengio"],
        "year": 2014,
        "field": "Natural Language Processing",
        "venue": "EMNLP",
        "keywords": ["Encoder-Decoder", "GRU", "Machine Translation", "RNN"],
        "section": "Abstract",
        "text": "This paper is a cornerstone of Neural Machine Translation (NMT), introducing two major innovations that became standard in the field. First, the authors propose a new neural network architecture called the RNN Encoder-Decoder. This model consists of two recurrent neural networks: an encoder that maps a variable-length source sequence into a fixed-length vector (a 'context' or 'thought' vector), and a decoder that maps that vector back into a variable-length target sequence. This was a significant departure from traditional Statistical Machine Translation (SMT), which relied on complex, hand-crafted features and separate models for translation and language modeling. The Encoder-Decoder framework allows for end-to-end training, where the model learns to maximize the conditional probability of the target sequence given the source. The second major innovation is the Gated Recurrent Unit (GRU). Inspired by the success of LSTMs, the GRU is a simplified gated mechanism designed to capture long-term dependencies while being computationally more efficient. Unlike LSTMs, which have three gates, the GRU has only two: a reset gate and an update gate. The reset gate determines how much of the previous state to forget, while the update gate determines how much of the new information to incorporate into the current state. The authors evaluated their model on the task of translating English to French. They showed that while the Encoder-Decoder could be used as a standalone translation system, it was particularly effective when used to 'rescore' the candidate translations produced by an existing SMT system. The experiments demonstrated that the RNN Encoder-Decoder significantly improved translation quality, especially for long and complex sentences. The paper also provided a detailed analysis of the learned hidden states, showing that the model's fixed-length vector captured meaningful semantic and syntactic properties of the source phrases. This work effectively proved that deep learning could handle the 'sequence-to-sequence' problem, opening the door for many other applications beyond translation, such as text summarization, dialogue systems, and even code generation. The Encoder-Decoder paradigm, combined with the later development of attention mechanisms, would eventually evolve into the Transformer, but the foundational idea of using one network to 'understand' and another to 'generate' started here."
    },
    {
        "paper_id": "paper_017",
        "title": "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift",
        "authors": ["Sergey Ioffe", "Christian Szegedy"],
        "year": 2015,
        "field": "Computer Vision",
        "venue": "ICML",
        "keywords": ["Batch Normalization", "Optimization", "Deep Learning", "Regularization"],
        "section": "Abstract",
        "text": "Training deep neural networks is a notoriously difficult and slow process. One of the primary reasons for this is a phenomenon the authors call 'Internal Covariate Shift.' As the parameters of a network change during training, the distribution of inputs to each internal layer also changes. This means that each layer must constantly adapt to a new input distribution, which requires lower learning rates and careful parameter initialization, significantly slowing down the optimization process. This paper introduces Batch Normalization, a simple yet transformative technique to address this problem. The core idea is to normalize the inputs to each layer for every mini-batch of training data, ensuring they have a mean of zero and a variance of one. To ensure the network can still represent any function, the authors also introduce two learnable parameters per layer—a scale and a shift—that allow the network to 'undo' the normalization if it's beneficial. By stabilizing the distribution of layer activations throughout training, Batch Normalization allows researchers to use much higher learning rates, which drastically accelerates the convergence of the model. Furthermore, Batch Normalization makes the network much less sensitive to the initial values of its parameters, which previously could mean the difference between a model that trains well and one that doesn't train at all. The authors also found that Batch Normalization acts as a form of regularization; by adding a small amount of noise to the activations (due to the mini-batch statistics), it reduces the need for other regularization techniques like Dropout. The researchers tested Batch Normalization on state-of-the-art image classification models on the ImageNet dataset. They found that they could achieve the same level of accuracy as the original models while using 14 times fewer training steps. When they pushed the models further, they surpassed the previous state-of-the-art performance by a significant margin. Today, Batch Normalization is a ubiquitous component in nearly all deep learning architectures, from ResNets to GANs. While there is still some debate in the academic community about the exact mechanism of why it works so well—whether it's reducing covariate shift or smoothing the optimization landscape—there is no denying its practical impact. It is one of the most effective tools for making deep learning models more robust, efficient, and easier to train."
    },
    {
        "paper_id": "paper_018",
        "title": "Show, Attend and Tell: Neural Image Caption Generation with Visual Attention",
        "authors": ["Kelvin Xu", "Jimmy Ba", "Ryan Kiros", "Kyunghyun Cho", "Aaron Courville", "Ruslan Salakhutdinov", "Richard Zemel", "Yoshua Bengio"],
        "year": 2015,
        "field": "Multimodal AI",
        "venue": "ICML",
        "keywords": ["Image Captioning", "Attention", "Multimodal", "Computer Vision"],
        "section": "Abstract",
        "text": "Generating a natural language description for an image—a task known as image captioning—requires a model to not only identify the objects in the scene but also understand their relationships and the overall context. This paper introduces a groundbreaking approach to this problem by incorporating a 'Visual Attention' mechanism into a neural captioning model. Inspired by how humans focus on specific parts of a scene when describing it, the authors propose a model that selectively 'looks' at different parts of an image as it generates each word of the caption. The architecture consists of a convolutional neural network (CNN) that acts as an encoder, extracting a set of feature vectors from different spatial locations in the image. These features are then passed to a recurrent neural network (RNN) decoder. At each time step, the decoder uses an attention mechanism to compute a weighted sum of the image features, focusing on the areas most relevant to the word it is about to generate. The paper explores two types of attention: 'soft' attention, which is a deterministic and differentiable approach that can be trained using standard backpropagation, and 'hard' attention, which is a stochastic approach that requires reinforcement learning (specifically the REINFORCE algorithm) to train. One of the most compelling aspects of this work is its interpretability. The authors provide visualizations showing the model's 'gaze' as it generates a caption; for instance, when generating the word 'bird,' the model's attention is clearly focused on the pixels containing the bird. This was one of the first times that the internal workings of a deep multimodal model were made transparent and intuitive. Evaluated on several benchmarks (Flickr8k, Flickr30k, and MS COCO), the attention-based model significantly outperformed previous state-of-the-art captioning systems. Beyond its performance, the paper proved that attention is a powerful tool for bridging the gap between vision and language. It paved the way for more advanced multimodal systems, such as Visual Question Answering (VQA) and video captioning, where the ability to dynamically attend to relevant information is crucial. The 'Show, Attend and Tell' framework remains a classic in the field, demonstrating how a simple biological intuition can lead to a sophisticated and highly effective artificial intelligence system."
    },
    {
        "paper_id": "paper_019",
        "title": "Contrastive Language-Image Pre-training (CLIP)",
        "authors": ["Alec Radford", "Jong Wook Kim", "Chris Hallacy", "Aditya Ramesh", "Gabriel Goh", "Sandhini Agarwal", "Girish Sastry", "Amanda Askell", "Pamela Mishkin", "Jack Clark"],
        "year": 2021,
        "field": "Multimodal AI",
        "venue": "ICML",
        "keywords": ["CLIP", "Contrastive Learning", "Multimodal", "Zero-Shot"],
        "section": "Abstract",
        "text": "Traditional computer vision models are limited by the closed-set nature of their training; they can only recognize the specific object categories they were explicitly taught during supervision. This paper introduces CLIP (Contrastive Language-Image Pre-training), a model that overcomes this limitation by learning visual concepts from natural language. Instead of using a human-labeled dataset like ImageNet, the authors pre-trained CLIP on a massive dataset of 400 million image-text pairs scraped from the internet. The model uses a simple yet powerful contrastive objective: given a batch of images and their corresponding captions, it is trained to maximize the cosine similarity between the correct image-text pairs while minimizing the similarity for all other possible pairs. This 'matching' task forces the model to learn a shared embedding space where images and their linguistic descriptions are aligned. The resulting model is incredibly versatile. Because it has learned to associate visual features with words, it can be used for any visual classification task in a 'zero-shot' manner. You simply provide the names of the classes you want to recognize, and CLIP will identify which image most closely matches each description. The authors tested CLIP on over 30 different computer vision benchmarks and found that it achieved performance comparable to state-of-the-art supervised models on many of them, without ever seeing a single labeled example from those specific datasets. Most impressively, CLIP is far more robust than traditional models to 'distribution shift.' For example, while a standard model might fail if it's shown a sketch of a dog instead of a real photo, CLIP's broad language-based training allows it to generalize easily to different artistic styles and environments. The paper also highlights CLIP's ability to perform complex tasks like OCR (Optical Character Recognition) and action recognition. By showing that natural language is a rich and sufficient source of supervision for vision, CLIP has fundamentally changed how we build and evaluate visual models. It is now used as a key component in many other state-of-the-art systems, including DALL-E 2 and Stable Diffusion, serving as the bridge that connects the world of pixels with the world of concepts. This work suggests that the future of AI lies in multimodal models that can learn from the vast, unstructured data available on the web, rather than relying on expensive and limited human-annotated datasets."
    },
    {
        "paper_id": "paper_020",
        "title": "Generative Pre-training (GPT)",
        "authors": ["Alec Radford", "Karthik Narasimhan", "Tim Salimans", "Ilya Sutskever"],
        "year": 2018,
        "field": "Generative AI",
        "venue": "OpenAI Technical Report",
        "keywords": ["GPT", "Pre-training", "Transformer", "Transfer Learning"],
        "section": "Abstract",
        "text": "Learning effective representations from unlabeled text has long been a challenge in Natural Language Processing, as most tasks have very little labeled data. This paper explores a powerful approach to this problem: using a two-stage training procedure. The first stage is an unsupervised 'Generative Pre-training' phase, where a large-scale transformer model is trained on a massive, diverse corpus of unlabeled text. The objective is simple: predict the next word in a sequence given the previous words. This autoregressive task forces the model to learn a deep understanding of syntax, semantics, and even some world knowledge. The second stage is 'Supervised Fine-tuning,' where the pre-trained model is adapted to a specific downstream task, such as sentiment analysis, question answering, or textual entailment, using a much smaller labeled dataset. The authors used a Transformer decoder as their base architecture, which, unlike the encoder-only models like BERT, is specifically designed for generation. The results were dramatic. GPT achieved significant improvements on 9 out of 12 benchmarks, setting new state-of-the-art results across a variety of domains. The paper demonstrates that the pre-trained model captures useful linguistic features that generalize well across different tasks. For instance, the model's performance on the CoLA (Corpus of Linguistic Acceptability) dataset showed it had learned a sophisticated understanding of English grammar. The researchers also found that the transformer architecture was more effective for this type of transfer learning than previous recurrent models like LSTMs. This work was a significant milestone because it shifted the focus of the NLP community away from task-specific architectures and toward general-purpose models that could be 'specialized' for any need. It laid the groundwork for the 'GPT' series of models, which would eventually lead to the massive GPT-3 and the widely known ChatGPT. By proving that generative pre-training is a viable and powerful path toward building intelligent systems, the authors helped usher in the current era of Foundation Models. The paper emphasizes the scalability of this approach, suggesting that as more data and more compute are applied to the pre-training phase, the resulting models will become increasingly capable and generalizable, a prediction that has been borne out in the years since its publication."
    }
]

#### Analyzing Token Counts in Our Dataset

Before we proceed with encoding our Research paper texts, let's check how many tokens each text contains. This is important because our encoder model has a 256-token limit, and we need to know if our descriptions exceed this limit.

We'll use the same tokenizer that our encoder uses to get accurate token counts:

In [13]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Count tokens for each text
for doc in documents:
  tokens = tokenizer.encode(doc["text"], add_special_tokens=False)
  print(f"{doc['title']}: {len(tokens)} tokens")

  # show if it exceeds
  if len(tokens) > 256:
    print(f"  -  Exceeds 256 token limit by {len(tokens) - 256} tokens")
  print()

Attention Is All You Need: 443 tokens
  -  Exceeds 256 token limit by 187 tokens

ImageNet Classification with Deep Convolutional Neural Networks: 476 tokens
  -  Exceeds 256 token limit by 220 tokens

BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding: 501 tokens
  -  Exceeds 256 token limit by 245 tokens

Deep Residual Learning for Image Recognition: 448 tokens
  -  Exceeds 256 token limit by 192 tokens

Generative Adversarial Networks: 482 tokens
  -  Exceeds 256 token limit by 226 tokens

Language Models are Few-Shot Learners: 507 tokens
  -  Exceeds 256 token limit by 251 tokens

Playing Atari with Deep Reinforcement Learning: 425 tokens
  -  Exceeds 256 token limit by 169 tokens

Mask R-CNN: 459 tokens
  -  Exceeds 256 token limit by 203 tokens

Mastering the Game of Go with Deep Neural Networks and Tree Search: 449 tokens
  -  Exceeds 256 token limit by 193 tokens

Learning Joint Adversarial-Learned Representations for Multimodal AI: 431 tokens
  - 

Since our texts are longer than 256 tokens, we'll need to chunk them into smaller pieces before encoding, which we'll implement in the next steps.

### 5. Setting Up Qdrant

Now we'll initialize our Qdrant vector search engine that will store our reserch paper embeddings and enable similarity searches.

We're using an in-memory mode for this demo, which means data is stored temporarily in RAM, and our data will be lost when the session ends.

> **<font color='red'>Warning:</font>** In this mode, no vector index is built; all searches are performed exactly.

In [14]:
client = QdrantClient(':memory:')
collection_name = 'research_papers'

### 6. Create the Collection

We’ll now create a collection in Qdrant designed to store multiple vector representations per document. We will test different chunking strategies.

First, we check whether the collection `research_papers` already exists and delete it to avoid conflicts.  Then we define the collection with three named vector fields: `fixed`, `sentence`, and `semantic`, each with the same embedding size and the distance metric.

In [15]:
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

embedding_dimension = encoder.get_sentence_embedding_dimension()
assert embedding_dimension is not None, "Embedding dimension cannot be None."

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        'fixed': models.VectorParams(size=embedding_dimension, distance=models.Distance.COSINE),
        'sentence': models.VectorParams(size=embedding_dimension, distance=models.Distance.COSINE),
        'semantic': models.VectorParams(size=embedding_dimension, distance=models.Distance.COSINE),
    }
)

True

### 7. Implementing Text Chunking Strategies

Since our texts exceed the 256-token "comprehension" limit of our encoder, we need to implement text chunking strategies to break texts into smaller, manageable pieces.

We'll use three different approaches to compare their effectiveness.

In [16]:
MAX_TOKENS = 256

def fixed_size_chunks(text, size=MAX_TOKENS):
  "Split text into fixed-size token chunks"
  tokens = tokenizer.encode(text, add_special_tokens=False)
  return [
      tokenizer.decode(tokens[i:i+size], add_special_tokens=False)
      for i in range(0, len(tokens), size)
  ]

def sentence_splitter(text, size=MAX_TOKENS):
  splitter = SentenceSplitter(chunk_size=size, chunk_overlap=40)
  return splitter.split_text(text)

def semantic_splitter(text):
  document = Document(text=text)

  semantic_splitter = SemanticSplitterNodeParser(
      buffer_size=1,
      breakpoint_percentile_threshold=95,
      embed_model=HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
  )
  nodes = semantic_splitter.get_nodes_from_documents([document])  # Pass list of Document objects
  return [n.get_content() for n in nodes]

Chunking Strategies:
- **Fixed-size chunks**: Splits text into raw 256-token blocks using the tokenizer. This method ignores sentence boundaries and is purely positional. Fast, but may split ideas in awkward places. Each chunk is exactly 256 tokens (or less for the final chunk).

- **Sentence chunks**: Uses `SentenceSplitter` from LlamaIndex to group sentences into 256-token chunks with 40-token overlap. This preserves syntactic boundaries and keeps sentences intact, offering a clean middle ground between structure and context. Sentences are grouped together until they reach the token limit.

- **Semantic chunks**: Uses `SemanticSplitterNodeParser` from LlamaIndex to create meaning-aware chunks based on the underlying content. It dynamically determines the best chunk boundaries using embedding-based scoring with a 95th percentile threshold and buffer size of 1. This approach creates chunks that are semantically coherent rather than just syntactically complete.

These three approaches will help us understand how different text segmentation methods affect search quality and relevance. The semantic approach should theoretically provide the most contextually relevant chunks for search, while fixed-size chunks offer the most predictable and uniform results.

### 8. Chunk, Embed, and Upload to Qdrant

We’ll now process all research paper texts by applying each of our three chunking strategies, embedding the resulting chunks, and uploading them to Qdrant.

For every research paper text, we generate:

- Fixed-size chunks: Chunks split purely by token count
- Sentence-based chunks: Sentences truncated to fit token limits
- Semantic chunks: Smart splits using sentence boundaries and overlap

Each chunk is embedded using our encoder and stored with metadata indicating:

- The original paper info (paper_id, title, author, year,...)
- The chunk’s text
- The chunking strategy used (fixed, sentence, or semantic)

In [17]:
points = []
idx = 0

for doc in documents:
  # Fixed-size
  for chunk in fixed_size_chunks(doc["text"]):
    points.append(models.PointStruct(
        id=idx,
        vector={'fixed': encoder.encode(chunk).tolist()},
        payload={**doc, "chunk": chunk, "chunking": "fixed"},
    ))
    idx += 1

  # Sentence
  for chunk in sentence_splitter(doc["text"]):
    points.append(models.PointStruct(
        id=idx,
        vector={'sentence': encoder.encode(chunk).tolist()},
        payload={**doc, "chunk": chunk, "chunking": "sentence"},
    ))
    idx += 1

  #Semantic
  for chunk in semantic_splitter(doc["text"]):
    points.append(models.PointStruct(
        id=idx,
        vector={'semantic': encoder.encode(chunk).tolist()},
        payload={**doc, "chunk": chunk, "chunking": "semantic"},
    ))
    idx += 1

client.upload_points(collection_name=collection_name, points=points)
print(f"Uploaded {idx} vectors.")



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Uploaded 127 vectors.


### 9. Run a Semantic Search Query

Now that our research paper chunks are embedded and indexed in Qdrant, let’s query the collection to find the most relevant matches for a given search intent.

In this example, we’ll search for documents related to “computer vision” using one of our three chunking strategies.

In [20]:
results = client.query_points(
    collection_name=collection_name,
    query=encoder.encode("Computer Vision").tolist(),
    using="fixed",
    limit=3,
)

for point in results:
  print(point)

('points', [ScoredPoint(id=65, version=0, score=0.4323329606803054, payload={'paper_id': 'paper_011', 'title': 'You Only Look Once: Unified, Real-Time Object Detection', 'authors': ['Joseph Redmon', 'Santosh Divvala', 'Ross Girshick', 'Ali Farhadi'], 'year': 2016, 'field': 'Computer Vision', 'venue': 'CVPR', 'keywords': ['YOLO', 'Object Detection', 'Real-time', 'CNN'], 'section': 'Abstract', 'text': "Before the introduction of YOLO (You Only Look Once), object detection systems were complex, multi-stage pipelines. Models like R-CNN used a 'proposal and classify' approach: they first generated thousands of potential bounding boxes in an image and then ran a heavy classifier on each box to identify objects. While accurate, these systems were incredibly slow and difficult to optimize because each component had to be trained separately. This paper presents a radically different approach: framing object detection as a single regression problem. YOLO uses a single neural network to predict b

In [21]:
for i, point in enumerate(results.points, 1):
    payload = point.payload
    print(
        f"{i}. {payload['title']} ({payload['year']})\n"
        f"   Score: {point.score:.4f}\n"
        f"   Chunking: {payload['chunking']}\n"
        f"   Chunk: {payload['chunk']}\n"
    )

1. You Only Look Once: Unified, Real-Time Object Detection (2016)
   Score: 0.4323
   Chunking: fixed
   Chunk: benefit is that yolo reasons globally about the image. because it sees the entire image during training and inference, it captures contextual information about classes and their appearances, which helps it avoid making background errors — a common problem for localized methods like fast r - cnn. while yolo ' s localization accuracy initially lagged behind state - of - the - art systems, especially for small objects, its speed and generalizability were unmatched. the authors showed that yolo generalizes better than other methods when moving from natural images to artwork, proving that it learns more robust features. this paper fundamentally changed the trajectory of object detection research, shifting the community ' s focus from pure accuracy to the critical trade - off between speed and performance. yolo has since gone through many iterations ( v2, v3, v4, etc. ) and remains

#### Define a Query Helper Function

To make it easier to run multiple searches with different chunking strategies, we’ll define a reusable helper function. It accepts a search query, the vector field to use ("fixed", "sentence", or "semantic"), and how many top results to return.

In [24]:
def search_and_print(query, vector_name, k=3):
    results = client.query_points(
        collection_name=collection_name,
        query=encoder.encode(query).tolist(),
        using=vector_name,  # 'fixed', 'sentence', or 'semantic'
        limit=k,
    )

    print(f"\nTop {k} results using '{vector_name}' chunks for query: '{query}'")
    for point in results.points:
        print(point.payload['title'], "| score:", point.score)

We can now call

In [25]:
search_and_print("Retrieval Augmented generation", "semantic")
search_and_print("auto encoders", "sentence")



Top 3 results using 'semantic' chunks for query: 'Retrieval Augmented generation'
DALL-E: Zero-Shot Text-to-Image Generation | score: 0.40470037337214115
Language Models are Few-Shot Learners | score: 0.3944824591033643
Generative Pre-training (GPT) | score: 0.3492993393152586

Top 3 results using 'sentence' chunks for query: 'auto encoders'
Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation | score: 0.5613250854350607
DALL-E: Zero-Shot Text-to-Image Generation | score: 0.541354960949801
An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale | score: 0.45064401925400344


#### Inspect Retrieved Chunks in Detail

Let's define a function to see exactly what text was retrieved.

In [26]:
def search_and_inspect(query, vector_name, k=3):
    results = client.query_points(
        collection_name=collection_name,
        query=encoder.encode(query).tolist(),
        using=vector_name,
        limit=k,
        with_payload=True,
    )

    print(f"\nTop {k} results using '{vector_name}' chunks for query: '{query}'\n")
    for i, point in enumerate(results.points, 1):
        payload = point.payload
        print(
            f"{i}. {payload['title']} ({payload['year']})\n"
            f"   Score: {point.score:.4f}\n"
            f"   Chunking: {payload['chunking']}\n"
            f"   Chunk: {payload['chunk']}\n"
        )

Now let's test the same query, "Computer vision", across all three chunking strategies

In [27]:
for strategy in ['fixed', 'sentence', 'semantic']:
    search_and_inspect('Computer vision', strategy)


Top 3 results using 'fixed' chunks for query: 'Computer vision'

1. You Only Look Once: Unified, Real-Time Object Detection (2016)
   Score: 0.4323
   Chunking: fixed
   Chunk: benefit is that yolo reasons globally about the image. because it sees the entire image during training and inference, it captures contextual information about classes and their appearances, which helps it avoid making background errors — a common problem for localized methods like fast r - cnn. while yolo ' s localization accuracy initially lagged behind state - of - the - art systems, especially for small objects, its speed and generalizability were unmatched. the authors showed that yolo generalizes better than other methods when moving from natural images to artwork, proving that it learns more robust features. this paper fundamentally changed the trajectory of object detection research, shifting the community ' s focus from pure accuracy to the critical trade - off between speed and performance. yolo has s

Comparing their output will give you a sense of which strategy retrieves more coherent or relevant segments.

### 10. Apply a Filter to Your Search Query

Let's include a filter `query_filter` that restricts the results to research papers where the `year` payload field is greater than or equal to 2017.



In [34]:
hits = client.query_points(
    collection_name=collection_name,
    query=encoder.encode("Computer Vision").tolist(),
    using="semantic",
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="year",
                range=models.Range(gte=2017)
            )
        ]
    ),
    limit=4,
    with_payload=True,
)

for point in hits.points:
    print(point.payload['title'], "| score:", point.score)

An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale | score: 0.3814387755996109
DALL-E: Zero-Shot Text-to-Image Generation | score: 0.36201208468359386
Mask R-CNN | score: 0.33883905650232626
Contrastive Language-Image Pre-training (CLIP) | score: 0.3285344009035097


### Grouping Results by Title

When working with chunked text, a regular vector search might return multiple high-scoring chunks from the same document.

Optionally, we can use `query_points_groups()` to group results by a field in the payload. In this case, the `title` of the Research paper, so that only the best-matching chunk from each movie is returned.



In [37]:
response = client.query_points_groups(
    collection_name=collection_name,
    query=encoder.encode("Computer Vision").tolist(),
    using="semantic",
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="year",
                range=models.Range(gte=2017)
            )
        ]
    ),
    group_by="title",       # group results by the 'title' field
    limit=4,               # number of unique titles to return
    group_size=1,          # max points per group
    with_payload=True,
)

for group in response.groups:
    print(group.id, "| score:", group.hits[0].score)

An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale | score: 0.3814387755996109
DALL-E: Zero-Shot Text-to-Image Generation | score: 0.36201208468359386
Mask R-CNN | score: 0.33883905650232626
Contrastive Language-Image Pre-training (CLIP) | score: 0.3285344009035097
